In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import pytesseract

In [2]:
import easyocr

In [12]:
model = YOLO("C:/Users/ARUNIMA KUNDU/Downloads/best (3).pt")

In [4]:
reader = easyocr.Reader(['en'])  # you can add multiple languages

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [5]:
import re

In [9]:
pattern = re.compile(r'[A-Z]{2} \d{3,4}')

In [17]:
cap = cv2.VideoCapture("C:/Users/ARUNIMA KUNDU/Downloads/traffic_video_original.mp4")
c=1
while True:
    ret,frame = cap.read()
    if ret == False:
        break
    # print("Frame",c)
    c+=1
    # img = cv2.resize(frame,(640,640))
    output = model.predict(frame,verbose=False)
    labels = output[0].boxes.cls
    coords = output[0].boxes.xyxy
    # print(coords)
    if len(labels)>0:
        for i in range(len(labels)):
            x1,y1,x2,y2 = int(coords[i][0]),int(coords[i][1]),int(coords[i][2]),int(coords[i][3])
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            region = frame[y1:y2,x1:x2]
            region = cv2.cvtColor(region,cv2.COLOR_BGR2Lab)
            region,_,_ = cv2.split(region)

            clahe = cv2.createCLAHE(clipLimit=3,tileGridSize=(8,8))
            cl_region = clahe.apply(region)
           
            if region is None:
                continue
            # extracted_number = pytesseract.image_to_string(region)
            results = reader.readtext(cl_region,paragraph=True)
            
            if len(results) == 1:
                txt = results[0][1]
                if pattern.fullmatch(txt):
                    cv2.putText(frame,results[0][1],(x1,y1-5),fontFace=cv2.FONT_HERSHEY_SIMPLEX,fontScale=1,color=(0,255,0),thickness=2)
                    
    img = cv2.resize(frame,(640,640))
    cv2.imshow("Output",img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    # print(coords)
cv2.destroyAllWindows()
cap.release()

In [20]:
import pytesseract

# Example path (adjust if different)
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'


In [5]:
img = cv2.resize(cv2.imread("F:/wallpapers/1_qre-gAVNTuazaUPvNw2w-Q.jpg"),(640,640))
o = model.predict(img)

# o = model.predict("F:/wallpapers/1_qre-gAVNTuazaUPvNw2w-Q.jpg")
# img = cv2.imread("F:/wallpapers/1_qre-gAVNTuazaUPvNw2w-Q.jpg")
l = o[0].boxes.cls
c = o[0].boxes.xyxy
for i in range(len(l)):
    cv2.rectangle(img,(int(c[i][0]),int(c[i][1])),(int(c[i][2]),int(c[i][3])),(0,255,2),2)
cv2.imshow("Result",img)
cv2.waitKey(0)
cv2.destroyAllWindows()


0: 640x640 (no detections), 351.5ms
Speed: 80.5ms preprocess, 351.5ms inference, 15.2ms postprocess per image at shape (1, 3, 640, 640)
